# EVE 310 - Lab 09: Batch processing — one building

**Module 3 | 10/22/2026**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThyanRevolter/eve310-fall-2026/blob/main/labs/lab09-batch-processing/notebooks/lab09-single-file.ipynb)

## Learning objectives

By the end of this lab you will be able to:

1. Load one campus water file and parse datetimes
2. Remove 3-sigma outliers
3. Draw a monthly box plot and save it

## Before you start

- **On your laptop:** run `uv sync` in the repository root, then launch this notebook with `uv run jupyter lab` (see `docs/setup.md`).
- **In Google Colab:** click *Copy to Drive* first, or your work is lost when the tab closes.
- Either way, run the setup cell below before anything else. It sets `DATA_DIR` and `FIGURES_DIR` to wherever this lab's files live.
- Work through the cells in order. Cells marked **Your turn** are for you to complete.


Campus building water files live in `DATA_DIR` with names like `water_DCP.csv`. The three-letter code after `water_` is the building.


## 0. Setup


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Course setup: run this cell first. It works on your laptop and in Google Colab.
LAB = "lab09-batch-processing"
try:
    from eve310 import setup_notebook
except ModuleNotFoundError:
    # Colab starts from a blank machine, so download the course files first.
    !test -d /content/eve310 || git clone -q --depth 1 https://github.com/ThyanRevolter/eve310-fall-2026.git /content/eve310
    import sys
    sys.path.insert(0, "/content/eve310/src")
    from eve310 import setup_notebook

DATA_DIR, FIGURES_DIR = setup_notebook(LAB)


## 1. Load one file


In [ ]:
file_path = DATA_DIR / 'water_DCP.csv'
water_df = pd.read_csv(file_path)
building = file_path.stem.replace('water_', '')
print(building)
water_df.head()


## 2. Worked example


In [ ]:
water_df['DateTime'] = pd.to_datetime(water_df['DateTime'])
water_df['Month'] = water_df['DateTime'].dt.month
col = 'Water ( Gallons )'

w_std = np.std(water_df[col], ddof=1)
w_mean = np.mean(water_df[col])
upper, lower = w_mean + 3 * w_std, w_mean - 3 * w_std
water_df = water_df.loc[(water_df[col] < upper) & (water_df[col] > lower)]

ax = water_df.boxplot(column=col, by='Month', grid=False, rot=45)
plt.xticks(
    range(1, 13),
    ['January', 'February', 'March', 'April', 'May', 'June',
     'July', 'August', 'September', 'October', 'November', 'December'],
)
plt.ylabel('Consumption (gallons)')
plt.xlabel('Month')
plt.title(f'{building} water consumption by month')
plt.suptitle('')
plt.savefig(FIGURES_DIR / f'{building}_water_boxplot.png', bbox_inches='tight', dpi=200)


## 3. Wrap-up

Next: loop over every CSV in `DATA_DIR` in `lab09-multiple-files.ipynb`.
